In [ ]:
import os
import sys

WORKDIR = os.environ["CONTAINER_WORK_DIR"]

os.chdir(WORKDIR)
sys.path.append(f"{WORKDIR}/test/modules/simul_whisper")

In [ ]:
import torch
import numpy as np
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.esic_v1 import ESICv1Dataset

In [ ]:
from simul_whisper.transcriber.config import AlignAttConfig
from simul_whisper.transcriber.simul_whisper import PaddedAlignAttWhisper, DEC_PAD

In [ ]:
dataset = ESICv1Dataset.load(Path(f"{WORKDIR}/test/performance_test/data/esic_train.json"))

In [ ]:
key, audio, text = dataset[0]

In [ ]:
cfg = AlignAttConfig(
    model_path="small",
    segment_length=3,
    frame_threshold = 12,
    language="en",
    buffer_len = 20,
    min_seg_len=0.0,
    if_ckpt_path=f"{WORKDIR}/test/modules/simul_whisper/cif_models/small.pt"
)
model = PaddedAlignAttWhisper(cfg)

In [ ]:
from simul_whisper.whisper.audio import N_FFT, HOP_LENGTH, SAMPLE_RATE

class Segment:
    def __init__(self, audio:np.ndarray, samples_to_read, samples_in_chunk):
        self.audio = torch.from_numpy(audio).float()
        self.audio_len_s = self.audio.shape[0] / SAMPLE_RATE
        self.samples_to_read = samples_to_read
        self.samples_in_chunk = samples_in_chunk
        self.buffer_len = samples_in_chunk - samples_to_read

    def __iter__(self):
        frames_in_chunk = self.audio[:self.samples_in_chunk]
        read_pointer = frames_in_chunk.shape[0]
        yield frames_in_chunk, (read_pointer >= self.audio.shape[0])
        while read_pointer < self.audio.shape[0]:
            frames_in_chunk = torch.cat(
                (frames_in_chunk[-self.buffer_len:], self.audio[read_pointer:read_pointer+self.samples_to_read]),
                dim=0
                )
            read_pointer += self.samples_to_read
            yield frames_in_chunk, (read_pointer >= self.audio.shape[0])


class SegmentWrapper(Segment):
    def __init__(self, audio:np.ndarray, segment_length):
        frames_to_read = int((segment_length * SAMPLE_RATE) / HOP_LENGTH)
        samples_to_read = frames_to_read * HOP_LENGTH
        samples_in_chunk = samples_to_read + N_FFT - HOP_LENGTH
        super().__init__(
            audio,
            samples_to_read=samples_to_read,
            samples_in_chunk=samples_in_chunk)

In [ ]:
hyp_list = []
segmented_audio = SegmentWrapper(audio, cfg.segment_length)
for seg_id, (seg, is_last) in enumerate(segmented_audio):
    new_toks = model.infer(seg, is_last)
    hyp_list.append(new_toks)
    hyp = torch.cat(hyp_list, dim=0)
    hyp = hyp[hyp < DEC_PAD]
    hyp = model.tokenizer.decode(hyp)
    print(hyp)
model.refresh_segment(complete=True)